# E3 Models on E4 Test Graphs

Transferability eval: e3-trained models (Llama-3.1-8B, trained on e3 data, smaller graphs)
evaluated against the larger e4 test graphs.

**Models evaluated:**
- `e3_llm` — plain LLM LoRA, `text_edge_list=present` (9 graphs, 90 tasks)
- `e3_rpearl` — graph-augmented (RPEARL), `text_edge_list=none` (5 graphs, 50 tasks — run was interrupted)

Results written by `scripts/eval_checkpoint_on_graphs.py` to `<checkpoint>/eval_logs/cross_eval/`.

In [ ]:
import json
import os
from glob import glob
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

In [ ]:
RESULTS_ROOT = "../outputs/e3_new_training_data"

MODEL_LABELS = {
    "e3_llm_llama-3.1-8b_r16_4bit_0sy9j5rz": "e3 LLM (text edge list)",
    "e3_rpearl_llm_llama-3.1-8b_r16_4bit_qmu8x2qu": "e3 RPEARL (graph-augmented)",
}

records = []
for path in sorted(glob(f"{RESULTS_ROOT}/*/eval_logs/cross_eval/*.json")):
    if "debug_graph" in path:
        continue
    with open(path) as f:
        d = json.load(f)
    ckpt_id = d.get("checkpoint", "").split("/")[-1]
    graph_name = os.path.basename(d.get("graph_file", path)).replace(".json", "")
    records.append({
        "model": MODEL_LABELS.get(ckpt_id, ckpt_id),
        "ckpt_id": ckpt_id,
        "graph": graph_name,
        "architecture": d.get("architecture", ""),
        "text_edge_list": d.get("text_edge_list", ""),
        "correct": d.get("num_correct", 0),
        "total": d.get("num_samples", 0),
        "accuracy": d.get("accuracy", 0.0),
    })

df = pd.DataFrame(records)
print(f"{len(df)} result files, {df['total'].sum()} total tasks")
df.head()

## Overall Accuracy by Model

In [ ]:
summary = (
    df.groupby(["model", "architecture", "text_edge_list"])
    .agg(correct=("correct", "sum"), total=("total", "sum"), graphs=("graph", "count"))
    .reset_index()
)
summary["accuracy"] = summary["correct"] / summary["total"]
summary["result"] = summary.apply(lambda r: f"{r['correct']}/{r['total']}", axis=1)
display(summary[["model", "architecture", "text_edge_list", "graphs", "result", "accuracy"]].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#4ec9b0", "#ce9178"]
bars = ax.bar(summary["model"], summary["accuracy"], color=colors[:len(summary)], width=0.5)
for bar, row in zip(bars, summary.itertuples()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{row.accuracy:.0%}\n({row.result})",
        ha="center", va="bottom", fontsize=10,
    )
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(0, 0.55)
ax.set_title("E3 Models on E4 Test Graphs — Overall Accuracy", fontsize=12)
ax.set_ylabel("Accuracy")
ax.set_xlabel("")
plt.xticks(rotation=10, ha="right")
plt.tight_layout()
plt.savefig("e3_on_e4_overall.png", dpi=150)
plt.show()

## Per-Graph Accuracy

In [ ]:
pivot = df.pivot(index="graph", columns="model", values="accuracy")
pivot = pivot.sort_index()
display(pivot.style.format("{:.0%}").background_gradient(axis=None, cmap="RdYlGn", vmin=0, vmax=1))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
models = df["model"].unique()
graphs = sorted(df["graph"].unique())
x = np.arange(len(graphs))
w = 0.35
bar_colors = ["#4ec9b0", "#ce9178"]

for i, model in enumerate(models):
    sub = df[df["model"] == model].set_index("graph")
    vals = [sub.loc[g, "accuracy"] if g in sub.index else np.nan for g in graphs]
    offset = (i - (len(models) - 1) / 2) * w
    ax.bar(x + offset, vals, width=w, label=model, color=bar_colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(graphs, rotation=35, ha="right", fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy")
ax.set_title("Per-Graph Accuracy — E3 Models on E4 Graphs")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig("e3_on_e4_per_graph.png", dpi=150)
plt.show()

## Sample-Level Pass/Fail Detail

Drill into per-sample correctness, formatted output, and turn counts.

In [ ]:
sample_records = []
for path in sorted(glob(f"{RESULTS_ROOT}/*/eval_logs/cross_eval/*.json")):
    if "debug_graph" in path:
        continue
    with open(path) as f:
        d = json.load(f)
    ckpt_id = d.get("checkpoint", "").split("/")[-1]
    graph_name = os.path.basename(d.get("graph_file", path)).replace(".json", "")
    model_label = MODEL_LABELS.get(ckpt_id, ckpt_id)
    for s in d.get("samples", []):
        trace = s.get("interaction_trace", [])
        sample_records.append({
            "model": model_label,
            "graph": graph_name,
            "task": s.get("task", "")[:80],
            "correct": s.get("correct", False),
            "formatted": s.get("formatted", False),
            "turns": len(trace),
            "terminated_by": s.get("terminated_by", ""),
        })

sdf = pd.DataFrame(sample_records)
print(f"{len(sdf)} samples")
sdf.head()

In [ ]:
# Format rate and turn distribution per model
agg = sdf.groupby("model").agg(
    n=("correct", "count"),
    correct=("correct", "sum"),
    formatted=("formatted", "sum"),
    avg_turns=("turns", "mean"),
    max_turns=("turns", "max"),
).reset_index()
agg["acc"] = agg["correct"] / agg["n"]
agg["fmt_rate"] = agg["formatted"] / agg["n"]
display(agg[["model","n","correct","acc","formatted","fmt_rate","avg_turns","max_turns"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Turn distribution
for model, grp in sdf.groupby("model"):
    axes[0].hist(grp["turns"], bins=range(0, grp["turns"].max() + 2), alpha=0.6, label=model)
axes[0].set_xlabel("# Turns")
axes[0].set_ylabel("# Samples")
axes[0].set_title("Turn Count Distribution")
axes[0].legend(fontsize=8)

# Terminated-by breakdown
term = sdf.groupby(["model", "terminated_by"]).size().unstack(fill_value=0)
term.plot(kind="bar", ax=axes[1], rot=10)
axes[1].set_title("Termination Reason")
axes[1].set_ylabel("# Samples")
axes[1].legend(fontsize=8, title="terminated_by")

plt.tight_layout()
plt.show()

## Failure Analysis

Correct vs formatted — are failures due to bad formatting or wrong reasoning?

In [ ]:
fail = sdf[~sdf["correct"]].copy()
fail["reason"] = fail.apply(
    lambda r: "formatted but wrong" if r["formatted"] else "bad format", axis=1
)
fail_summary = fail.groupby(["model", "reason"]).size().unstack(fill_value=0)
display(fail_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
fail_summary.plot(kind="bar", ax=ax, rot=10, color=["#f48771", "#858585"])
ax.set_title("Failure Breakdown by Model")
ax.set_ylabel("# Failed Samples")
ax.legend(title="failure type", fontsize=9)
plt.tight_layout()
plt.show()

## Notes

- **e3 LLM (32.2%)** outperforms **e3 RPEARL (8.0%)** on the e4 graphs — consistent with e3 in-distribution results.
- RPEARL eval was only run on 5/9 graphs (interrupted); results are partial.
- Answer-key regex (`^(...)$`) requires exact short-form match — verbose correct answers count as failures. True accuracy may be higher.
- E4 graphs are larger than e3 training graphs; both models see a drop vs e3 in-distribution results, as expected.
- The RPEARL model uses `text_edge_list=none` (graph embedding replaces text edge list); LLM uses `text_edge_list=present`.